August 6th update: 
Goal for today: turn your RFM features into customer segments using K-means clustering, and log the model with MLflow so it's tracked properly — this is the "ML bridge" piece of the project

In [0]:
rfm_df = spark.read.format("delta").load("/Volumes/workspace/default/raw_uploads/gold/rfm_features")
rfm_df.show(5)

#why: reading it fresh from Delta (rather than reusing yesterday's in-memory rfm_df) confirms the persisted table is actually correct and complete - this is a good habit to get into, not just to correct in yesterday's session. its also exactly what a real pipeline would do: the ML step reads from a stable table, not from another notebook's memory

understand why rfm needs scaling before clustering:

why: recency is in days (can be 300+), frequency is a small integer (mostly 1-5), monetary is in currency (can be 700+)

why this matters: k-means measures distance between points. if one feature has values in the hundreds and another is just 1-5, the large-scale feature will completey dominate the distnace salculation - the models would essentially cluster on monetary alone and ignore frequency. scsling puts all three features on comparable footing o each contributes fairly to the clustering

In [0]:
# spark's ml library needs features conbined into a single vector column, not separate columns

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["recency_days", "frequency", "monetary"],
    outputCol="features_raw"
)

rfm_vectorized = assembler.transform(rfm_df)
rfm_vectorized.select("customer_unique_id", "features_raw").show(5, truncate=False)

#why: this is a spark ML convention, not a data change - VectorAssembler just packages our three numeric columns into one features_raw columbn that downstream ML stages expect as input

In [0]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(rfm_vectorized)
rfm_scaled = scaler_model.transform(rfm_vectorized)
rfm_scaled.select("customer_unique_id", "features").show(5, truncate=False)

#why: this directly solves the step 2 problem - StandardScaler transforms each feature to mean 0 and standard deviation 1, so recency, frequency, and monetary are all on that same scale ebfore clustering

In [0]:
from pyspark.ml.clustering import KMeans

kmeans = KMeans(featuresCol="features", predictionCol="segment", k=4, seed=42)
kmeans_model = kmeans.fit(rfm_scaled)

rfm_clustered = kmeans_model.transform(rfm_scaled)
rfm_clustered.groupBy("segment").count().orderBy("segment").show()

#why k=4: four segments is a classic, defensible RFM starting point - something like "champions/loyal/at-risk/lost" - and its small enough to be interpretable in a README without needing deep statistical justiciation. 
#seed=42 makes results reproducible if we rerun the cell
#what to look for: the groupBy("segment").count() output should show a reasonably sized group for each of the 4 segments 0 not one giant cluster and three nearly-empty ones. if its wildly imbalanced, the worth a second lok (could mean k should change, or a feature needs different scaling)

In [0]:
import pyspark.sql.functions as F
rfm_clustered.groupBy("segment").agg(
    F.round(F.avg("recency_days"), 1).alias("avg_recency"),
    F.round(F.avg("frequency"), 2).alias("avg_frequency"),
    F.round(F.avg("monetary"), 2).alias("avg_monetary"),
    F.count("*").alias("customer_count")
).orderBy("segment").show()

#why: this is the step that turns "i ran k-means" into "i built customer segments that answer a real question." look at the averages per segment and givbe each one a plain- english label - e.g. low recency + high frequency + high monetary = "champions", high recency + low frequency + low monetary = "at-risk/lost", write these labels down: you'll want them in your README and segmentation write-up tomorrow

In [0]:
import mlflow

with mlflow.start_run(run_name="rfm_kmeans_segmentation"):
    mlflow.log_param("k", 4)
    mlflow.log_param("features", "recency_days, frequency, monetary")
    mlflow.log_param("scaling", "StandardScaler (mean=0, std=1)")
    
    wssse = kmeans_model.summary.trainingCost
    mlflow.log_metric("wssse", wssse)
    
    # Save the model natively instead of mlflow.spark.log_model
    model_path = "/Volumes/workspace/default/raw_uploads/models/kmeans_rfm_model"
    kmeans_model.write().overwrite().save(model_path)
    mlflow.log_param("model_path", model_path)
    
    print("Run logged. WSSSE:", wssse)
    print("Model saved to:", model_path)

In [0]:
rfm_clustered.groupBy("segment").agg(
    F.round(F.avg("recency_days"), 1).alias("avg_recency"),
    F.round(F.avg("frequency"), 2).alias("avg_frequency"),
    F.round(F.avg("monetary"), 2).alias("avg_monetary"),
    F.count("*").alias("customer_count")
).orderBy("segment").show()

#

In [0]:
for seg in range(4):
    print(f"--- Segment {seg} sample ---")
    rfm_clustered.filter(F.col("segment") == seg).select(
        "customer_unique_id", "recency_days", "frequency", "monetary"
    ).show(5)

#why: confirms indivudal row actually match the segment-level averages - e.g., segment 1's samole should show real customers with 2+ orders, not just an average artifact

In [0]:
rfm_clustered.select(
    "customer_unique_id", "recency_days", "frequency", "monetary", "segment"
).write.format("delta").mode("overwrite").save("/Volumes/workspace/default/raw_uploads/gold/customer_segments")

print("Customer segments written successfully")

#persists the finished, labeled output so tomorrow's write-up (and any future dashboard) can reference it directly without rerunning clustering